# Сравнение методов интерпертаций
В этом ноутбуке мы проверим эффективность предложенного метода. Будем сравнивать методы на изображениях из ImageNet на модели VGG16.
В первой части мы проверим метод на качество (Insection и Delection). 
Во второй части проверим скорость работы различных методов.

In [ ]:
!pip install grad-cam
!pip install captum

In [1]:
import torch
import torchvision
import torch.nn.functional as F
from torchvision import  models, transforms
from torchvision.models import resnet50
import torch.nn.functional as F
from PIL import Image
import requests
from io import BytesIO
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os

from captum.attr import GuidedBackprop, IntegratedGradients, LayerGradCam
from captum.attr import visualization as viz

from pytorch_grad_cam import ScoreCAM
from pytorch_grad_cam import HiResCAM
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

## Load images

In [2]:
image_directory = "/kaggle/input/datasets/tusonggao/imagenet-validation-dataset/imagenet_validation"

image_paths = []

if os.path.exists(image_directory):
    for root, dirs, files in os.walk(image_directory):
        for filename in files:
            if filename.lower().endswith(('.jpeg', '.jpg', '.png')):
                image_paths.append(os.path.join(root, filename))
else:
    print(f"Error: Directory {image_directory} does not exist.")

print(f"Loaded {len(image_paths)} images.")

Loaded 50000 images.


## Load ImageNet Class Labels

In [3]:
IMAGENET_LABELS_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
response = requests.get(IMAGENET_LABELS_URL)
imagenet_labels = [line.strip() for line in response.text.split('\n') if line.strip()]

 ## Реализация GradScoreCAM

In [12]:
import tqdm
from pytorch_grad_cam.base_cam import BaseCAM

class GradScoreCAM(BaseCAM):
    def __init__(self, model, target_layers, reshape_transform=None, topk_ratio=0.05):
        super(GradScoreCAM, self).__init__(model,
                                       target_layers,
                                       reshape_transform=reshape_transform,
                                       uses_gradients = True)
        self.topk_ratio = topk_ratio
        self.sel_activations = torch.Tensor().to(self.device)

    def get_cam_weights(self,
                        input_tensor,
                        target_layer,
                        targets,
                        activations_full,
                        grads):

        grads = torch.from_numpy(grads).to(self.device)
        with torch.no_grad():

            upsample = torch.nn.UpsamplingBilinear2d(
                size=input_tensor.shape[-2:]
            )

            activation_tensor_full = torch.from_numpy(activations_full).to(self.device) #[B, C, H, W]

            # importance scores for each batch
            hires_scores = torch.sum(activation_tensor_full * grads, dim=(2, 3)) #[B, C]
            B, C = hires_scores.shape

            # select k the most important activation maps
            k = max(1, int(C * self.topk_ratio))
            topk_indices = torch.topk(hires_scores, k=k, dim=1).indices  # [B, k]

            # collect maps for each batch
            selected_activations_list = []
            for b in range(B):
                selected = activation_tensor_full[b, topk_indices[b]]  # [k, H, W]
                selected_activations_list.append(selected)
            self.sel_activations = torch.stack(selected_activations_list)  # [B, k, H, W]

            # upsample
            upsampled = upsample(self.sel_activations)
            maxs = upsampled.view(upsampled.size(0), upsampled.size(1), -1).max(dim=-1)[0]
            mins = upsampled.view(upsampled.size(0), upsampled.size(1), -1).min(dim=-1)[0]

            maxs, mins = maxs[:, :, None, None], mins[:, :, None, None]
            upsampled = (upsampled - mins) / (maxs - mins + 1e-8)

            input_tensors = input_tensor[:, None,
                                         :, :] * upsampled[:, :, None, :, :]

            if hasattr(self, "batch_size"):
                BATCH_SIZE = self.batch_size
            else:
                BATCH_SIZE = 16

            scores = []
            for target, tensor in zip(targets, input_tensors):
                for i in range(0, tensor.size(0), BATCH_SIZE):
                    batch = tensor[i: i + BATCH_SIZE, :]
                    outputs = [target(o).cpu().item()
                               for o in self.model(batch)]
                    scores.extend(outputs)
            scores = torch.Tensor(scores)
            scores = scores.view(self.sel_activations.shape[0], self.sel_activations.shape[1])
            weights = torch.nn.Softmax(dim=-1)(scores).numpy()
            return weights

    def get_cam_image(
        self,
        input_tensor: torch.Tensor,
        target_layer: torch.nn.Module,
        targets: list,
        activations_full: torch.Tensor,
        grads: torch.Tensor,
        eigen_smooth: bool = False) -> np.ndarray:

        weights = self.get_cam_weights(input_tensor, target_layer, targets, activations_full, grads)
        activations_for_weighted_sum = self.sel_activations.cpu().detach().numpy()

        # 2D conv
        if len(activations_for_weighted_sum.shape) == 4:
            weighted_activations = weights[:, :, None, None] * activations_for_weighted_sum
        # 3D conv
        elif len(activations_for_weighted_sum.shape) == 5:
            weighted_activations = weights[:, :, None, None, None] * activations_for_weighted_sum
        else:
            raise ValueError(f"Invalid activation shape. Get {len(activations_for_weighted_sum.shape)}.")

        if eigen_smooth:
            cam = get_2d_projection(weighted_activations)
        else:
            cam = weighted_activations.sum(axis=1)
        return cam



## Сравнение методов интепертации на метриках Insection AUC и Delection AUC

1.   Grad-CAM
2.   HiResCAM
3.   ScoreCAM
4.   GradScoreCAM



In [11]:
# load VGG16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.vgg16(
    weights=models.VGG16_Weights.IMAGENET1K_V1).to(device)
model.eval()
target_layers = [model.features[-1]]

cam_methods = {
    "gradcam": GradCAM(model=model, target_layers=target_layers),
    "hirescam": HiResCAM(model=model, target_layers=target_layers),
    "scorecam": ScoreCAM(model=model, target_layers=target_layers),
    "gradscorecam": GradScoreCAM(
        model=model,
        target_layers=target_layers,
        topk_ratio=0.1
    )
}

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

### Сравним методы на 1000 изображениях

In [17]:
import random

random.seed(42)

selected_paths = random.sample(image_paths, 1000)


results = {
    method: {
        "insertion_auc": [],
        "deletion_auc": []
    }
    for method in cam_methods
}

CHECKPOINT_PATH = "cam_results_checkpoint.json"

start_idx = 0

if os.path.exists(CHECKPOINT_PATH):

    with open(CHECKPOINT_PATH, "r") as f:
        checkpoint = json.load(f)

    results = checkpoint["results"]
    start_idx = checkpoint["last_index"] + 1

    print(f"Resuming from image {start_idx}")

Resuming from image 1


In [10]:
def auc(curve):
    return np.trapezoid(curve)

def normalize_map(cam):

    cam = cam - cam.min()

    if cam.max() > 0:
        cam = cam / cam.max()

    return cam

In [18]:
@torch.no_grad()
def compute_deletion_auc(
    model,
    input_tensor,
    cam,
    target_class,
    device,
    steps=50
):

    model.eval()
    cam = normalize_map(cam)
    _, _, H, W = input_tensor.shape
    cam = cv2.resize(cam, (W, H))
    # flatten
    flat_cam = cam.flatten()
    # most important first
    indices = np.argsort(-flat_cam)
    total_pixels = H * W
    pixels_per_step = total_pixels // steps
    # original image
    deleted = input_tensor.clone()

    scores = []

    for i in range(steps):
        start = i * pixels_per_step
        end = min((i + 1) * pixels_per_step, total_pixels)
        current_indices = indices[start:end]
        rows = current_indices // W
        cols = current_indices % W
        # delete pixels
        deleted[:, :, rows, cols] = 0
        output = model(deleted)
        prob = torch.softmax(output, dim=1)[0, target_class]
        scores.append(prob.item())

    return auc(scores)

In [19]:
@torch.no_grad()
def compute_insertion_auc(
    model,
    input_tensor,
    cam,
    target_class,
    device,
    steps=50
):

    model.eval()
    cam = normalize_map(cam)
    _, _, H, W = input_tensor.shape
    cam = cv2.resize(cam, (W, H))
    flat_cam = cam.flatten()
    indices = np.argsort(-flat_cam)
    total_pixels = H * W
    pixels_per_step = total_pixels // steps

    # black image
    inserted = torch.zeros_like(input_tensor).to(device)
    scores = []

    for i in range(steps):
        start = i * pixels_per_step
        end = min((i + 1) * pixels_per_step, total_pixels)
        current_indices = indices[start:end]
        rows = current_indices // W
        cols = current_indices % W
        # insert pixels
        inserted[:, :, rows, cols] = input_tensor[:, :, rows, cols]
        output = model(inserted)
        prob = torch.softmax(output, dim=1)[0, target_class]
        scores.append(prob.item())

    return auc(scores)

In [ ]:
import gc
import json

for idx in tqdm.tqdm(range(start_idx, len(selected_paths))):

    path = selected_paths[idx]

    try:
        pil_img = Image.open(path).convert("RGB")
        input_tensor = preprocess(pil_img).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(input_tensor)
            target_class = output.argmax(dim=1).item()

        for method_name, cam_method in cam_methods.items():
            try:
                grayscale_cam = cam_method(
                    input_tensor=input_tensor,
                    targets=None
                )[0]

                insertion_auc = compute_insertion_auc(
                    model,
                    input_tensor,
                    grayscale_cam,
                    target_class,
                    device=device
                )

                deletion_auc = compute_deletion_auc(
                    model,
                    input_tensor,
                    grayscale_cam,
                    target_class,
                    device=device
                )

                results[method_name]["insertion_auc"].append(
                    float(insertion_auc)
                )

                results[method_name]["deletion_auc"].append(
                    float(deletion_auc)
                )

                print(
                    f"[{idx}] {method_name} "
                    f"INS={insertion_auc:.4f} "
                    f"DEL={deletion_auc:.4f}"
                )

            except Exception as e:
                print(f"ERROR in {method_name} on image {idx}: {e}")

            finally:
                torch.cuda.empty_cache()
                gc.collect()

        if idx % 50 == 0:

            checkpoint = {
                "last_index": idx,
                "results": results
            }

            with open(CHECKPOINT_PATH, "w") as f:
                json.dump(checkpoint, f)
            print(f"Checkpoint saved at image {idx}")

    except Exception as e:
        print(f"FAILED IMAGE {idx}: {e}")

    finally:
        del input_tensor
        torch.cuda.empty_cache()
        gc.collect()


with open("final_cam_results.json", "w") as f:
    json.dump(results, f)

In [21]:
for method in results:
    ins_mean = np.mean(results[method]["insertion_auc"])
    del_mean = np.mean(results[method]["deletion_auc"])
    print(method, ins_mean, del_mean)

gradcam 24.0150625190036 4.483157907082424
hirescam 23.298748744138667 4.646350265884174
scorecam 24.62012325679551 4.317468336944483
gradscorecam 24.563059377189987 4.329288521280754


In [18]:
def benchmark_cam_method(
    method,
    image_paths,
    device="cuda",
    warmup_runs=5,
    runs_per_image=5,
):

    assert torch.cuda.is_available(), "CUDA is not available"
    method.model.eval()
    timings = []

    with Image.open(image_paths[0]).convert("RGB") as pil_img:
        warmup_tensor = preprocess(pil_img).unsqueeze(0).to(device)
    print("Warmup...")
    for _ in range(warmup_runs):
        _ = method(warmup_tensor)

    del warmup_tensor
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.synchronize()

    starter = torch.cuda.Event(enable_timing=True)
    ender = torch.cuda.Event(enable_timing=True)
    print("Benchmarking...")

    for path in tqdm(image_paths):
        with Image.open(path).convert("RGB") as pil_img:
            input_tensor = preprocess(pil_img).unsqueeze(0).to(device)
            
        for _ in range(runs_per_image):
            torch.cuda.synchronize()
            starter.record()
            cam = method(input_tensor)
            ender.record()
            torch.cuda.synchronize()
            curr_time = starter.elapsed_time(ender)
            timings.append(curr_time)
            del cam

        del input_tensor

        torch.cuda.empty_cache()
        gc.collect()

    timings = np.array(timings)
    mean_time = timings.mean()
    std_time = timings.std()

    print(f"Mean inference time: {mean_time:.3f} ms")
    print(f"Std inference time : {std_time:.3f} ms")
    print(f"FPS                : {1000 / mean_time:.2f}")

    return mean_time, std_time

In [ ]:
from tqdm import tqdm
import random

selected_paths = random.sample(image_paths, 50)


results_time = {
    method: {
        "mean time": 0,
        "std time": 0
    }
    for method in cam_methods
}


for method_name, cam_method in cam_methods.items():
    try:
        mean_time, std_time = benchmark_cam_method(
            method=cam_method,
            image_paths=selected_paths,
            device="cuda",
            warmup_runs=2,
            runs_per_image=2,
        )
        
        results_time[method_name]["mean time"] = mean_time
        results_time[method_name]["std time"] = std_time

        print(
            f"{method_name}:\n"
            f"mean time = {mean_time:.4f}\n"
            f"std time = {std_time:.4f}"
        )
    
    except Exception as e:
        print(f"ERROR in {method_name}: {e}")